# Track A — Mid-latitude extremes workflow demo

**Reflective SAI Uncertainty Database · Track A: mid-latitude extreme winter weather response to SAI**

This notebook demonstrates the `sai_extremes` open-science workflow end to end:
1. discover the available **models × scenarios** from the data catalog;
2. open standardized daily fields through the **pluggable I/O layer** (production);
3. compute the full **24-index ETCCDI set** (fixed + percentile-based);
4. quantify the **SAI vs HiLLA** contrast and **inter-model spread**.

It runs **anywhere** using the portable NumPy path on synthetic data. The
Cloud Hub cells (xarray + real data) are shown for reference and run unchanged
once `config/catalog.yaml` points at the Hub.


In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np, matplotlib.pyplot as plt
import sai_extremes as se
from sai_extremes import synthetic, pipeline, regions
from sai_extremes.indices import INDEX_REGISTRY
print('sai_extremes', se.__version__)
print('Indices:', len(se.list_indices()))


## 1. The data catalog

Model × scenario availability is declarative. Note MIROC has no HiLLA entry
(it is not in the multi-model HiLLA ensemble of Duffey et al., 2026).


In [ ]:
from sai_extremes.catalog import load_catalog
cat = load_catalog('../config/catalog.yaml')
for m in cat.models():
    avail = [s for s in cat.scenarios() if cat.is_available(m, s)]
    print(f'{m:8s} -> {avail}')
print('\nAvailable pairs:', cat.available_pairs())


## 2. Production I/O (Cloud Hub) — reference only

On the Hub, one call returns a standardized daily `xarray.Dataset`
(CF names, °C, mm day⁻¹, lon in [-180,180)). Backend (NetCDF/S3/Zarr) is
chosen by the catalog — the analysis code never changes.
```python
from sai_extremes.io import open_daily
ds = open_daily('CESM', 'G6-1.5K-SAI', ['tasmax','tasmin','pr'])
from sai_extremes.pipeline import run_model_scenario
indices = run_model_scenario('CESM', 'G6-1.5K-SAI', base_ds=ref_ds)
indices.to_netcdf('etccdi_CESM_G6-1.5K-SAI.nc')   # CF-1.10
```


## 3. Portable path — generate synthetic daily data and compute indices

The synthetic generator injects the documented *sign* of the response
(tropical-injection SAI warms the Eurasian winter and dries the
Mediterranean more than HiLLA) so the demo figures are interpretable.


In [ ]:
daily = synthetic.generate_daily('CESM', 'G6-1.5K-SAI', years=(2060,2066), nlat=8, nlon=16)
base  = synthetic.generate_daily('CESM', 'baseline',    years=(2050,2056), nlat=8, nlon=16)
res = pipeline.compute_all_indices_numpy(daily, base=base)
for k in ['FD','TNn','CSDI','TX90p','Rx5day','R95p']:
    f = res['fields'][k]
    print(f"{k:7s} {INDEX_REGISTRY[k].long_name:42s} mean={np.nanmean(f):7.2f} {INDEX_REGISTRY[k].units}")


## 4. SAI vs HiLLA and inter-model spread

Compute one winter-relevant index (frost days, FD) over the Eurasian box for
every available model × scenario, and show the spread.


In [ ]:
def region_mean(field, lat, lon, region):
    lat_s,lat_n,lon_w,lon_e = regions.REGIONS[region]
    la=(lat>=lat_s)&(lat<=lat_n); lo=(lon>=lon_w)&(lon<=lon_e)
    sub=field[np.ix_(la,lo)]; w=np.cos(np.deg2rad(lat[la]))[:,None]*np.ones((1,lo.sum()))
    m=~np.isnan(sub); return float(np.sum(sub[m]*w[m])/np.sum(w[m]))

models=cat.models(); scen=['G6-1.5K-SAI','G6-1.5K-HiLLA']; index='FD'; region='Eurasia_high'
vals={s:[] for s in scen}
for m in models:
    for s in scen:
        if not cat.is_available(m,s): vals[s].append(np.nan); continue
        d=synthetic.generate_daily(m,s,years=(2060,2064),nlat=8,nlon=16)
        b=synthetic.generate_daily(m,'baseline',years=(2050,2054),nlat=8,nlon=16)
        f=pipeline.compute_all_indices_numpy(d,base=b,indices=[index])['fields'][index]
        vals[s].append(np.nanmean([region_mean(f[i],d['lat'],d['lon'],region) for i in range(f.shape[0])]))

x=np.arange(len(models)); w=0.38
fig,ax=plt.subplots(figsize=(8,4))
for k,s in enumerate(scen): ax.bar(x+(k-0.5)*w, vals[s], w, label=s)
ax.set_xticks(x); ax.set_xticklabels(models); ax.set_ylabel('days'); ax.legend()
ax.set_title('Frost days (FD), Eurasia 50–70°N — SAI vs HiLLA (synthetic demo)'); plt.show()


## 5. Next steps in the project

* Weeks 4–6 ("what"): run this index set on the **real** Cloud Hub data for
  both scenarios and all models → Deliverable 2 dataset.
* Weeks 7–8 ("why"): jet / storm-track / blocking diagnostics + composite
  anomalies to explain the extremes changes and quantify model spread.
* Weeks 9–10: fold the quantified signal into the revised **Extratropical
  Circulation** uncertainty entry (Deliverable 3).


---
# Part B — Weeks 7-8: circulation diagnostics (the "why")

The extremes indices above tell us *what* happens to mid-latitude winter
weather. To explain *why*, we diagnose the large-scale circulation that drives
those extremes: the **eddy-driven jet**, the **storm track**, atmospheric
**blocking** (Tibaldi & Molteni, 1990), and the **North Atlantic Oscillation**.

These use daily `ua` (850 hPa wind), `zg` (500 hPa height) and `psl`. Here we
drive them with synthetic Euro-Atlantic circulation fields whose scenario
signal follows the literature: tropical/subtropical-injection SAI gives a more
**positive NAO**, a **poleward, stronger jet**, and **less blocking** than HiLLA
(which has weaker tropical stratospheric heating). On the Cloud Hub, swap the
synthetic generator for `open_daily(..., ['ua','zg','psl'])` — the diagnostics
are unchanged.


In [ ]:
from sai_extremes import diagnostics as D
import numpy as np

def djf_mask(doy):
    return (doy <= 59) | (doy >= 335)   # DJF on a 365-day calendar

def boxmean(field, lat, lon, la0, la1, lo0, lo1):
    la = (lat>=la0)&(lat<=la1); lo = (lon>=lo0)&(lon<=lo1)
    return field[:, la][:, :, lo].mean(axis=(1,2))

scenarios = ['G6-1.5K-SAI', 'G6-1.5K-HiLLA']
circ = {s: synthetic.generate_circulation_daily('CESM', s, years=(2060,2064)) for s in scenarios}
base_circ = synthetic.generate_circulation_daily('CESM', 'baseline', years=(2050,2054))
lat, lon = base_circ['lat'], base_circ['lon']
print('grid', lat.size, 'x', lon.size, '| fields:', list(base_circ['units']))


### B1. Eddy-driven jet latitude & speed (Woollings et al., 2010)

DJF-mean, Atlantic-sector (60°W–0°) low-level zonal-wind profile; jet latitude
= latitude of the maximum (parabolic-refined), jet speed = the maximum.


In [ ]:
def jet(d):
    m = djf_mask(d['doy']); sec = (lon>=-60)&(lon<=0)
    prof = d['ua'][m][:, :, sec].mean(axis=(0,2))
    return D.jet_lat_speed(prof, lat)

for s in scenarios:
    jl, js = jet(circ[s]); jl0, js0 = jet(base_circ)
    print(f'{s:16s} jet_lat={jl:5.1f}N (Δ{jl-jl0:+.1f})  jet_speed={js:5.1f} m/s (Δ{js-js0:+.1f})')


### B2. North Atlantic Oscillation (station-based)

Azores-minus-Iceland SLP, with each station **standardized against the
baseline run** so the SAI/HiLLA means are comparable (a self-standardized
index would have zero mean by construction).


In [ ]:
def stations(d):
    az = boxmean(d['psl'], lat, lon, 36,40,-28,-20)
    ic = boxmean(d['psl'], lat, lon, 63,70,-25,-10)
    return az, ic

az0, ic0 = stations(base_circ)
az0m, az0s, ic0m, ic0s = az0.mean(), az0.std(), ic0.mean(), ic0.std()
for s in scenarios:
    az, ic = stations(circ[s]); m = djf_mask(circ[s]['doy'])
    nao = ((az-az0m)/az0s - (ic-ic0m)/ic0s)[m].mean()
    print(f'{s:16s} NAO index (DJF mean, baseline-standardized) = {nao:+.2f}')


### B3. Atmospheric blocking frequency (Tibaldi & Molteni, 1990)

Fraction of DJF days with a reversed meridional Z500 gradient at 60°N over the
Euro-Atlantic sector (a positive NAO / stronger jet suppresses blocking).


In [ ]:
def blocking(d, lo0=-10, lo1=30):
    m = djf_mask(d['doy']); jlon = (lon>=lo0)&(lon<=lo1)
    z = d['zg'][m][:, :, jlon]
    return float(np.mean([D.tm_blocking_frequency(z[:, :, j], lat) for j in range(z.shape[2])]))

for s in scenarios:
    print(f'{s:16s} blocking frequency = {blocking(circ[s]):.3f}  (baseline {blocking(base_circ):.3f})')


### B4. Storm-track intensity (transient eddy activity)

Temporal std of high-pass-filtered DJF Z500 over the North Atlantic — a
synoptic storm-track proxy.


In [ ]:
def storm(d, la0=45,la1=65,lo0=-60,lo1=0):
    m = djf_mask(d['doy']); la=(lat>=la0)&(lat<=la1); lo=(lon>=lo0)&(lon<=lo1)
    z = d['zg'][m][:, la][:, :, lo]
    vals = [D.storm_track_intensity(z[:, i, j]) for i in range(z.shape[1]) for j in range(z.shape[2])]
    return float(np.nanmean(vals))

for s in scenarios:
    print(f'{s:16s} storm-track intensity (m) = {storm(circ[s]):.1f}  (baseline {storm(base_circ):.1f})')


### B5. Summary figure — jet shift and blocking, SAI vs HiLLA


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
vals_jl = [jet(circ[s])[0] for s in scenarios] + [jet(base_circ)[0]]
vals_bf = [blocking(circ[s]) for s in scenarios] + [blocking(base_circ)]
vals_st = [storm(circ[s]) for s in scenarios] + [storm(base_circ)]
labels = scenarios + ['baseline']
cols = ['#1f77b4','#ff7f0e','#888888']
for ax,(title,vals,unit) in zip(axes,[('Jet latitude',vals_jl,'°N'),('Blocking frequency',vals_bf,'fraction'),('Storm-track intensity',vals_st,'m')]):
    ax.bar(labels, vals, color=cols); ax.set_title(title); ax.set_ylabel(unit)
    ax.tick_params(axis='x', rotation=20); ax.grid(axis='y', alpha=0.3)
fig.suptitle('Weeks 7-8 circulation diagnostics: SAI vs HiLLA (synthetic demo)')
fig.tight_layout(); plt.show()


### Interpretation

Read together, B1–B5 are the *mechanism* behind the Part A extremes: a more
positive NAO and a stronger, poleward-shifted jet under tropical/subtropical
SAI steer milder maritime air over northern Eurasia (→ fewer frost days,
weaker cold extremes) and shift storm-track precipitation northward (wetter
Northern Europe, drier Mediterranean), with reduced blocking. HiLLA, with less
tropical stratospheric heating, shows a muted version. **Quantifying how
consistently the four models reproduce this chain — and how it differs between
SAI and HiLLA — is exactly the inter-model-spread result that feeds the revised
Extratropical Circulation uncertainty entry (Deliverable 3).**
